In [1]:
# =========================================
# 04 - BERT Embeddings (IMPROVED & THESIS READY)
# =========================================

import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

tqdm.pandas()

# =========================================
# 1. PATHS
# =========================================

BASE_PATH = r"C:\Users\Neda\Desktop\personality_llm"

FEATURES_PATH = os.path.join(BASE_PATH, "data", "features")
EMBEDDINGS_PATH = os.path.join(BASE_PATH, "data", "embeddings")
MODEL_PATH = os.path.join(BASE_PATH, "models", "parsbert_model")

os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)

# =========================================
# 2. LOAD DATA
# =========================================

train_df = pd.read_csv(os.path.join(FEATURES_PATH, "train_features.csv"))
test_df = pd.read_csv(os.path.join(FEATURES_PATH, "test_features.csv"))
kamtera_df = pd.read_csv(os.path.join(FEATURES_PATH, "kamtera_features.csv"))

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Kamtera:", kamtera_df.shape)

# =========================================
# 3. DEVICE SETUP
# =========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================
# 4. LOAD MODEL (SAFE VERSION)
# =========================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

model = AutoModel.from_pretrained(MODEL_PATH, ignore_mismatched_sizes=True)
model.to(device)
model.eval()

print("✔ BERT model loaded")

# =========================================
# 5. MEAN POOLING (MASK-AWARE)
# =========================================

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

# =========================================
# 6. EMBEDDING GENERATOR (ROBUST)
# =========================================

def get_embeddings(texts, batch_size=16, max_length=256):
    all_embeddings = []

    texts = [str(t) for t in texts]

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            hidden_states = outputs.last_hidden_state
            pooled = mean_pooling(hidden_states, attention_mask)

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# =========================================
# 7. TRAIN EMBEDDINGS
# =========================================

print("\nExtracting train embeddings...")
train_embeddings = get_embeddings(train_df["clean_text"].tolist())

print("Train shape:", train_embeddings.shape)

# =========================================
# 8. TEST EMBEDDINGS
# =========================================

print("\nExtracting test embeddings...")
test_embeddings = get_embeddings(test_df["clean_text"].tolist())

print("Test shape:", test_embeddings.shape)

# =========================================
# 9. KAMTERA (CONTROLLED SAMPLING)
# =========================================

KAMTERA_SAMPLE_SIZE = min(10000, len(kamtera_df))
kamtera_sample = kamtera_df.sample(n=KAMTERA_SAMPLE_SIZE, random_state=42)

print("\nKamtera sample:", kamtera_sample.shape)

kamtera_embeddings = get_embeddings(kamtera_sample["clean_text"].tolist())

print("Kamtera embeddings shape:", kamtera_embeddings.shape)

# =========================================
# 10. SAVE EMBEDDINGS
# =========================================

np.save(os.path.join(EMBEDDINGS_PATH, "train_bert.npy"), train_embeddings)
np.save(os.path.join(EMBEDDINGS_PATH, "test_bert.npy"), test_embeddings)
np.save(os.path.join(EMBEDDINGS_PATH, "kamtera_bert_sample.npy"), kamtera_embeddings)

# =========================================
# 11. SAVE LABELS
# =========================================

np.save(
    os.path.join(EMBEDDINGS_PATH, "train_labels.npy"),
    train_df["label"].values
)

# =========================================
# 12. OPTIONAL META INFO (VERY IMPORTANT FOR THESIS)
# =========================================

meta = {
    "embedding_dim": train_embeddings.shape[1],
    "train_size": train_embeddings.shape[0],
    "test_size": test_embeddings.shape[0],
    "kamtera_size": kamtera_embeddings.shape[0],
    "model": "ParsBERT",
    "max_length": 256,
    "pooling": "mean_pooling"
}

pd.DataFrame([meta]).to_csv(
    os.path.join(EMBEDDINGS_PATH, "embedding_meta.csv"),
    index=False
)

# =========================================
# 13. FINAL SUMMARY
# =========================================

print("\n====================")
print("FINAL SUMMARY")
print("====================")

print("Train embeddings:", train_embeddings.shape)
print("Test embeddings:", test_embeddings.shape)
print("Kamtera embeddings:", kamtera_embeddings.shape)

print("\n✔ DONE - BERT embedding pipeline complete")

MODEL_PATH: C:\Users\Neda\Desktop\personality_llm\models\parsbert_model
Train: (800, 28)
Test: (800, 27)
Kamtera: (107280, 27)
Device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: C:\Users\Neda\Desktop\personality_llm\models\parsbert_model
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✔ BERT model loaded

Extracting train embeddings...


100%|██████████| 50/50 [00:19<00:00,  2.51it/s]


Train shape: (800, 768)

Extracting test embeddings...


100%|██████████| 50/50 [00:22<00:00,  2.24it/s]


Test shape: (800, 768)

Kamtera sample: (10000, 27)


100%|██████████| 625/625 [03:03<00:00,  3.41it/s]

Kamtera embeddings shape: (10000, 768)

FINAL SUMMARY
Train embeddings: (800, 768)
Test embeddings: (800, 768)
Kamtera embeddings: (10000, 768)

✔ DONE - BERT embedding pipeline complete
